In [0]:
# 1) Leer la landing (el "sistema origen")
from pyspark.sql import functions as F

SCHEMA = "credito_riesgo_lab"
TABLA_LANDING = f"workspace.{SCHEMA}.landing_clientes"
TABLA_BRONZE  = f"workspace.{SCHEMA}.clientes_bronze"

landing = spark.table(TABLA_LANDING)
print(f"Landing: {landing.count()} filas")
landing.printSchema()

In [0]:
# 2) BRONZE = copia fiel del origen + metadato de ingesta
# Regla de la capa bronze: NO se limpia nada. Solo añadimos _fecha_ingesta,
# que permite saber cuándo llegaron los datos (auditoría y reprocesado).
bronze = (
    landing
    .withColumn("_fecha_ingesta", F.current_timestamp())
)

(
    bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_BRONZE)
)

n_landing = spark.table(TABLA_LANDING).count()
n_bronze  = spark.table(TABLA_BRONZE).count()
print(f"Bronze: {n_bronze} filas (landing: {n_landing} → mismas filas, copia 1:1)")

In [0]:
# 3) Vista rápida: bronze es idéntica a la landing, incluidos duplicados y nulos
# La LIMPIEZA es trabajo de la capa silver (notebook 02)
display(spark.table(TABLA_BRONZE).limit(5))

# 01 · Capa BRONZE — Ingesta cruda

**Qué hace una capa bronze en la industria:** capturar los datos del origen **tal cual**, sin limpiar nada, añadiendo solo metadatos de ingesta (`_fecha_ingesta`). Es la *copia fiel* del sistema origen, por si hay que reprocesar.

**Mapping con la industria:** aquí sería una tarea de Lakeflow Connect / Auto Loader; lo mantenemos en PySpark para que veas la mecánica.

> Lee `landing_clientes` (creada por el bootstrap 00) y escribe `clientes_bronze`.